In [ ]:
"""1.This cell contains the code loading,checking and reshaping the dataset for further processing""" 

import torch 
import numpy as np

#Loading Nmpy Arrays from the dataset
raw = np.load("flowers_data.npy")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") #Fallback to CPU if GPU is not available
if(device.type == "cuda"):
    print("GPU is available")
    print("Converting Arrays to Tensors . . . .")
    #Converting the numpy arrays to torch tensors for faster computing
converted_x_label = torch.from_numpy(raw['X']).float() #W have converted to float because its necessary for further calculation accuracy
converted_y_label = torch.from_numpy(raw['Y']).float()
print(converted_x_label.shape,converted_x_label.device)
print(converted_y_label.shape,converted_y_label.device)
print("Tensors Created!")

#So the shape becomes (N, H, W, C) where N is the number of samples, H is the height of the image, W is the width of the image and C is the number of channels. But the model we are using expects the input shape to be (N, C, H, W). So we need to permute the dimensions of the tensor to match the input shape of the model.
permuted = torch.permute(converted_x_label, (0, 3, 1, 2)) #Permuting the dimensions of the tensor to match the input shape of the model


GPU is available
Converting Arrays to Tensors . . . .
torch.Size([11197, 150, 150, 3]) cpu
torch.Size([11197]) cpu
Tensors Created!


In [17]:
"""This cell is used to shuffle the X and Y labels so that the model is 
actually learning and not memorizing the whole dataset

"""
indices = torch.randperm(len(permuted))
X_shuffled = permuted[indices] / 255.0
Y_shuffled = converted_y_label[indices].long().squeeze()


In [18]:
#This cell splits the shuffled data into Training ans Test sets
#80% of the data is used for training and 20% for testing
train_split = int(0.8 * len(X_shuffled))
X_train = X_shuffled[:train_split]
Y_train = Y_shuffled[:train_split]
X_test = X_shuffled[train_split:]
Y_test = Y_shuffled[train_split:] 
print(len(X_train),len(Y_train),len(X_test),len(Y_test))



8957 8957 2240 2240


In [19]:
from torch.utils.data import DataLoader,TensorDataset
batch__size = 64
train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch__size, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, Y_test), batch_size=batch__size, shuffle=False)
print(f"Batches created for Train Data:{len(train_loader)} and Test Data: {len(test_loader)}")


Batches created for Train Data:140 and Test Data: 35


In [20]:
from torch import nn
import torch.optim as optim

class FlowerClassifier(nn.Module):
    def __init__(self):
        super().__init__()

        #Neural Network Initilization
        self.flatten = nn.Flatten()

        self.layer_1 = nn.Linear(in_features = 67500,out_features =128,bias=True)
        self.relu_1 = nn.ReLU()
        self.layer_2 = nn.Linear(in_features=128,out_features=64,bias=True)
        self.relu_2 = nn.ReLU()
        self.layer_3 = nn.Linear(in_features=64,out_features=32,bias=True)
        self.relu_3 = nn.ReLU()

        #Output Layer
        self.out = nn.Linear(in_features = 32, out_features=7)

    def forward(self,x):
        x = self.flatten(x)
        x = self.relu_1(self.layer_1(x))
        x = self.relu_2(self.layer_2(x))
        x = self.relu_3(self.layer_3(x))
        x = self.out(x)
        return x

model = FlowerClassifier().to(device)
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(model)

FlowerClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (layer_1): Linear(in_features=67500, out_features=128, bias=True)
  (relu_1): ReLU()
  (layer_2): Linear(in_features=128, out_features=64, bias=True)
  (relu_2): ReLU()
  (layer_3): Linear(in_features=64, out_features=32, bias=True)
  (relu_3): ReLU()
  (out): Linear(in_features=32, out_features=7, bias=True)
)


In [23]:
epochs = 20
for i in range(epochs):
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward Pass
        output = model(images)
        loss = loss_function(output, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
train_loss = running_train_loss / total_train
train_acc = (correct_train / total_train) * 100

    # ==========================
    # 2. TESTING (Every Epoch)
    # ==========================
model.eval()  # Freeze layers for evaluation
running_test_loss = 0.0
correct_test = 0
total_test = 0

with torch.inference_mode():  # Disable Autograd to save VRAM and time
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        test_outputs = model(images)
        t_loss = loss_function(test_outputs, labels)

        running_test_loss += t_loss.item() * images.size(0)
        _, test_preds = torch.max(test_outputs, 1)
        correct_test += (test_preds == labels).sum().item()
        total_test += labels.size(0)

test_loss = running_test_loss / total_test
test_acc = (correct_test / total_test) * 100

    # ==========================
    # 3. PRINT PROGRESS
    # ==========================
print(f"Epoch [{epoch+1:02d}/{epochs:02d}] | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")


NameError: name 'running_train_loss' is not defined